# Модели и конвейеры scikit-learn

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи самостоятельно, затем обратитесь к разделу решений.

## Уроки по этим темам

Материал этого ноутбука можно изучать по отдельным урокам:

- [Обучение и проверка в scikit-learn](../../python/ml-tools/training-evaluation/)
- [Преобразования и конвейеры](../../python/ml-tools/pipelines/)


## Чему научимся

Собирать подготовку данных и модель в один конвейер, подбирать параметр на обучающих разбиениях и отдельно измерять качество. Нужны pandas и идея KNN.

## Единый интерфейс

В scikit-learn `fit(X, y)` обучает по признакам и ответам, `predict(X)` выдаёт предсказания. Преобразователь учится через `fit`, а применяет выученное правило через `transform`. Например, масштабирование запоминает среднее и меру разброса каждого обучающего признака, затем использует их без пересчёта. Метод вызывается у объекта через точку: `model.fit(X, y)` обучает конкретную модель с именем `model`.

Таблица `X` имеет форму «объекты × признаки», `y` — один ответ на объект. Исключите из признаков правильный ответ, идентификатор и служебный признак разбиения. Хороший результат на обучении не заменяет проверку на новых объектах.

## Разбиение и метрика

Отделим тест — часть, которую не используем для выбора модели. Во время разработки подбираем параметры на обучающей части с внутренней **перекрёстной проверкой**: делим эту часть на несколько групп, по очереди оставляем одну для проверки, а на остальных обучаемся. Сравниваем средние оценки по всем таким проверкам. Тест в них не участвует. После выбора один раз измеряем качество на тесте. Для классификации доля верных ответов `accuracy` — понятная начальная метрика, но при редком классе она может вводить в заблуждение.

Пример ниже создаёт 24 вымышленных объекта. Размах не меньше 22 см задаёт класс 1, меньший — класс 0; масса и бумага нужны для демонстрации подготовки. Это простая задача для проверки кода, а не реалистичная оценка модели полёта.

`train_test_split` делит признаки и ответы согласованно, возвращая обучающие и тестовые части. `test_size=0.25` выделяет четверть объектов в тест. `stratify=y` сохраняет доли классов настолько близко, насколько позволяют целые числа объектов, `random_state=17` фиксирует начальное состояние случайного разбиения. Случайное разделение подходит не всем задачам: повторные измерения одного объекта надо держать вместе, временные данные проверять на будущем. Иначе тест может оказаться слишком лёгким.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

X = pd.DataFrame({
    "span_cm": list(range(10, 34)),
    "mass_g": [8., 9., 10., 11., None, 12.] * 4,
    "paper": ["тонкая", "плотная"] * 12,
})
y = np.array([0] * 12 + [1] * 12)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=17,
)
assert len(X_train) == 18 and len(X_test) == 6


## Разобранный пример: Pipeline и ColumnTransformer

`Pipeline` выполняет именованные шаги последовательно. `ColumnTransformer` применяет разные преобразования к разным столбцам. `SimpleImputer(strategy="median")` заполнит числовые пропуски медианой: центральным значением отсортированного ряда, а при чётной длине — средним двух центральных. Затем `StandardScaler` вычтет обучающее среднее и разделит на обучающее стандартное отклонение. Для чисел $x_1,\ldots,x_n$ со средним $m$ это $s=\sqrt{\frac1n\sum_i(x_i-m)^2}$: корень из среднего квадрата отклонений. Например, у `[1, 3]` среднее 2 и $s=1$, поэтому получим `[-1, 1]`. Постоянный обучающий столбец становится нулевым; деление на ноль библиотека не выполняет. Категории бумаги превратим в отдельные числовые индикаторы через one-hot кодирование: например, «тонкая» и «плотная» станут двумя столбцами, а строка получит 1 в столбце своей категории и 0 в другом.

`handle_unknown="ignore"` даёт нулевые индикаторы для неизвестной категории, позволяя применить модель к новым данным. Это техническое правило, а не доказательство качества на новых категориях.

`GridSearchCV` сравнивает кандидатов на трёх внутренних разбиениях. В каждом разбиении `Pipeline` заново обучает заполнение, масштабирование и KNN только на внутренней обучающей части. Двойное подчёркивание в `knn__n_neighbors` указывает параметр шага `knn`. После выбора лучший конвейер автоматически обучается на всей `X_train`.

Список шагов `Pipeline` состоит из пар `(имя_шага, объект_шага)`. Для `ColumnTransformer` каждый элемент — тройка `(имя, преобразователь, список_столбцов)`. Имена задаём сами и затем используем для настройки параметров. При `fit` конвейер учит преобразователи по порядку и передаёт подготовленные данные модели; при `predict` применяет уже выученные преобразования.

`OneHotEncoder(..., sparse_output=False)` возвращает обычный плотный массив чисел, в котором хранятся все ячейки, включая нули. `KNeighborsClassifier` создаёт классификатор KNN, его параметр `n_neighbors` задаёт число соседей. `StratifiedKFold(n_splits=3, ...)` задаёт три группы с близкими долями классов: на каждом шаге из 18 обучающих объектов 12 используются для обучения и 6 для внутренней проверки. `shuffle=True` предварительно перемешивает строки.

`GridSearchCV` перебирает значения 1, 3 и 5; `scoring="accuracy"` выбирает долю верных ответов. `best_params_` после обучения хранит выбранные параметры. Функция `accuracy_score(правильные_ответы, предсказания)` вычисляет эту долю. Для сравнения `DummyClassifier(strategy="most_frequent")` запоминает наиболее частый обучающий класс и выдаёт его каждому объекту — это постоянный baseline.

In [ ]:
numeric = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
prepare = ColumnTransformer([
    ("numbers", numeric, ["span_cm", "mass_g"]),
    ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["paper"]),
])
model = Pipeline([
    ("prepare", prepare),
    ("knn", KNeighborsClassifier()),
])
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=17)
search = GridSearchCV(model, {"knn__n_neighbors": [1, 3, 5]}, cv=cv, scoring="accuracy")
search.fit(X_train, y_train)
predicted = search.predict(X_test)
assert predicted.shape == y_test.shape
print(search.best_params_)
print("test accuracy:", accuracy_score(y_test, predicted))
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
assert accuracy_score(y_test, baseline.predict(X_test)) == 0.5


Ожидаем параметр из `{1, 3, 5}`, шесть предсказаний и baseline 0.5. Точный результат модели напечатает ноутбук: шесть тестовых объектов дают очень грубую оценку. Если после просмотра теста менять параметры, тест становится частью разработки и потребуется новая независимая проверка.

## Как обнаружить утечку

Опасный порядок: заполнить пропуски и масштабировать всю таблицу, затем разделить её. Даже без ответов преобразование уже использовало тест. Безопасный порядок: разделить исходные строки, затем передать обучение конвейеру. Сам конвейер не защищает от признака, вычисленного по правильным ответам, или одинаковых объектов по обе стороны разбиения.

Для справки: [типичные ошибки scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html).

## Задачи

### 1. Базовая

Чем отличаются `fit` и `predict`?

<details>
<summary>Подсказка</summary>

Один метод меняет обученное состояние, другой выдаёт ответы.

</details>



### 2. Базовая

Что означает `knn__n_neighbors`?

<details>
<summary>Подсказка</summary>

Две части задают шаг и его настройку.

</details>



### 3. Прикладная

Проверьте, что число предсказаний совпадает с числом тестовых объектов и каждый ответ — 0 или 1.

<details>
<summary>Подсказка</summary>

Используйте длину и множество.

</details>



### 4. Прикладная

Подайте модели объект с новой категорией бумаги и пропуском массы.

<details>
<summary>Подсказка</summary>

Создайте таблицу с теми же именами столбцов.

</details>



### 5. Олимпиадная

Почему медиану нужно учить внутри каждого разбиения GridSearchCV, а не один раз на всей X_train?

<details>
<summary>Подсказка</summary>

Внутренняя проверочная часть тоже должна оставаться невидимой подготовке.

</details>



### 6. Олимпиадная

В наборе 99% ответов класса 0. Модель всегда выдаёт 0 и имеет accuracy 0.99. Что проверить до объявления успеха?

<details>
<summary>Подсказка</summary>

Сравните с простейшим baseline и посчитайте ошибки по классам.

</details>



## Итоги и самопроверка

Нарисуйте путь данных от разбиения до финальной метрики. Объясните, где именно учатся медиана, масштабы и модель. Проверьте baseline прежде, чем усложнять решение.


## Решения задач

### Задача 1

`fit` учится по данным, `predict` применяет обученную модель.

### Задача 2

Параметр числа соседей у шага с именем `knn`.

### Задача 3

In [ ]:
assert len(predicted) == len(X_test)
assert set(predicted).issubset({0, 1})


### Задача 4

In [ ]:
new_object = pd.DataFrame({"span_cm": [18], "mass_g": [None], "paper": ["новая"]})
new_answer = search.predict(new_object)
assert new_answer.shape == (1,)


Преобразование успешно, но надёжность такого предсказания требует отдельной проверки.

### Задача 5

Иначе статистика внутренних проверочных строк попадёт в подготовку кандидатов и оценка перекрёстной проверки будет зависеть от невидимых данных.

### Задача 6

Такая модель совпадает с baseline и совсем не распознаёт класс 1. Нужны распределение классов, ошибки по каждому классу и метрика, соответствующая цели. Подробный выбор метрик рассматривается в маршруте ML.